# Quantize and Evaluate with PLENA

This notebook takes you from a fp16 HuggingFace model to a quantized,
benchmarked one — showing the full PLENA software workflow end to end.

We use **`unsloth/Llama-3.2-1B`** throughout the demo. 

## What you'll learn

- **The MX quantization recipe** — block-scaled integer/floating point formats expressed
  as a single TOML file, Based on Mase Library `quantize_module_transform_pass`.
- **Progressive recipes** — RTN, then `[gptq]` for weights quantization, then `[rotation_search]` for activation quatnization,.
- **Three benchmark demos** — perplexity, lm-eval, HumanEval+.
- **Save and reload** — the quantized model is a standard HF checkpoint.

## Live-execution timings

Each cell is sized to run in a reasonable demo slot on one GPU. The
`[gptq]` and `[rotation_search]` passes cache to disk, so reruns finish
in seconds once the first execution completes.


## MX quantization refresher

MX (Microscaling) formats store a tensor as low-precision **elements**
grouped into fixed-size **blocks** that share a single power-of-two
**scale** (Rouhani et al. 2023; OCP MX standard). PLENA's configurable MX
format follows this single-level scaling scheme and is what we quantize
weights, activations, and KV cache into.

![MX data formats: MXFP and MXINT, each block sharing one power-of-two scale](figures/mx_annotated.png)

*Figure adapted from the PLENA paper. MXFP elements carry sign +
exponent + mantissa; MXINT elements carry sign + mantissa. Both share a
single power-of-two scale per block, parameterised by the tunable tuple
$(M, E, S, B)$ for MXFP and $(M, S, B)$ for MXINT.*

### The format tuple

We describe an MX data format as a tuple

$$\tau = (d,\, b,\, B)$$

where $d$ is the element datatype (`INT` for MXINT, `minifloat` for
MXFP), $b$ is the element bit-width, and $B$ is the block size. So
$\tau = (\texttt{INT},\, 4,\, 16)$ is **MXINT4** with block size 16,
and $\tau = (\texttt{minifloat},\, 4,\, 16)$ is **MXFP4** with the
same block size. Every element in a block shares one scale $s$ and one
zero-point $z$; we use symmetric quantization throughout, so $z = 0$.

### Quantizing a block

Given a high-precision tensor $\mathbf{W}$ partitioned into blocks
$w \in \mathbb{R}^{B}$, the shared scale for each block is derived
from its absmax against the format's representable maximum:

$$s \;=\; \frac{\max |w|}{\max_{\tau}}$$

Each element is then projected into the low-precision grid by scaling,
round-to-nearest, and clipping to the representable range:

$$w_{\tau} \;=\; \mathrm{clip}\!\left(\mathrm{RTN}\!\left(\tfrac{w}{s}\right),\ \min_{\tau},\ \max_{\tau}\right)$$

Dequantization reconstructs an approximation of the original block
using the same shared scale:

$$Q(w;\, s,\, \tau) \;=\; s \cdot w_{\tau}$$

For integer MX formats the representable range is
$[\min_{\tau},\, \max_{\tau}] = [-(2^{b-1}-1),\, 2^{b-1}-1]$; for
MXFP it is set by the minifloat's $(E, M)$ encoding.

### What PLENA exposes in TOML

| Format    | Element                 | Width knobs |
|-----------|-------------------------|-------------|
| **MXInt** | Signed integer          | `weight_width`, `data_in_width` |
| **MXFP**  | Mini-float (exp + frac) | `weight_exponent_width` / `weight_frac_width`, `data_in_exponent_width` / `data_in_frac_width` |

Block size is set via `weight_block_size` / `data_in_block_size`. The
typical "MX4" setting used throughout this tutorial is 4-bit MXINT elements
with block size 32.

The full TOML schema is in our documentation
[Quantization configs](https://aicrosssim.github.io/PLENA_Software/reference/toml-reference/).


## The PTQ Algorithm
We'll quantize Llama-3.2-1B three ways and measure the accuracy at each step:

| Recipe | Adds                              | Where the gain comes from |
|--------|-----------------------------------|---------------------------|
| **A**  | MXInt4 round-to-nearest           | Baseline — the simplest possible quantization |
| **B**  | + `[gptq]` Hessian-aware calib    | Re-derives W4 weights so quantization error is *output-aware*, not just element-aware |
| **C**  | + `[rotation_search]` per-matmul  | Picks which matmuls benefit from an online Hadamard rotation, on a per-model basis |

Recipe C *reuses Recipe B's GPTQ checkpoint*.


## Recipe A — MXInt4 RTN

The simplest recipe: every linear projection in a decoder block gets W4 + A4
MXInt with block size 32.

Each section header is a regex that matches **module names**; the body
says how to quantize the matching modules. The regex targets Llama-style
layer names, so the recipe ports to Llama-2/3, Qwen, and most modern
decoders.


In [9]:
# Local artifacts (recipe TOMLs, calibration caches, saved
# checkpoints) land under output/.
!mkdir -p output/checkpoints

In [ ]:
%%writefile output/recipe_a_rtn.toml
by = "regex_name"

# Attention projections: W4 + A4
["model\\.layers\\.\\d+\\.self_attn\\.(q|k|v|o)_proj"]
name = "mxint"
weight_block_size = 32
weight_width = 4
data_in_block_size = 32
data_in_width = 4

# MLP projections: W4 + A4
["model\\.layers\\.\\d+\\.mlp\\.(gate|up|down)_proj"]
name = "mxint"
weight_block_size = 32
weight_width = 4
data_in_block_size = 32
data_in_width = 4

Overwriting output/recipe_a_rtn.toml


### Apply the recipe in Python

This is what every `eval_*` CLI does under the hood. Three steps:

1. **Load** the model with the eager attention backend.
2. **Parse** the TOML into a pass-args dict.
3. **Transform** the model in place — every matching `nn.Linear` is
   swapped for a `LinearMXInt` module that quantizes weights and
   activations on every call.


In [13]:
import torch
from quant_eval.utils import setup_model
from quant_eval.quantize import load_quant_config
from chop.passes.module.transforms import quantize_module_transform_pass

# 1. Load fp16 baseline on CPU
tokenizer, model = setup_model(
    "unsloth/Llama-3.2-1B",
    model_parallel=False,
    dtype=torch.float16,
    device=None,                 
    attn_implementation="eager",
)
model.eval()

# 2. Read the recipe
pass_args = load_quant_config("output/recipe_a_rtn.toml")

# 3. Apply the quantization pass, then move to GPU
model, _ = quantize_module_transform_pass(model, pass_args)
model.to("cuda:0")

Loading weights: 100%|██████████| 146/146 [00:00<00:00, 455.19it/s]


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128004)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): LinearMXInt(in_features=2048, out_features=2048, bias=False)
          (k_proj): LinearMXInt(in_features=2048, out_features=512, bias=False)
          (v_proj): LinearMXInt(in_features=2048, out_features=512, bias=False)
          (o_proj): LinearMXInt(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): LinearMXInt(in_features=2048, out_features=8192, bias=False)
          (up_proj): LinearMXInt(in_features=2048, out_features=8192, bias=False)
          (down_proj): LinearMXInt(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
   

### What changed?

Every projection in a decoder block was a plain `torch.nn.Linear` before;
after the transform, those slots hold `LinearMXInt`. 

For Llama-3.2-1B: **7 linear projections × 16 layers = 112** `LinearMXInt`
modules, with the embedding, lm_head, RMSNorms, and attention internals
left at full precision. You can extend the recipe to cover these modules
too — see the [TOML reference](https://aicrosssim.github.io/PLENA_Software/reference/toml-reference/)
for ready-to-use selectors covering the attention QK/AV matmuls, softmax,
RMSNorm, and the embedding / lm_head.


In [14]:
from collections import Counter

cls_counts = Counter(type(m).__name__ for _, m in model.named_modules())
for name, n in cls_counts.most_common(8):
    print(f"  {name:30s}  ×{n}")

  LinearMXInt                     ×112
  LlamaRMSNorm                    ×33
  LlamaDecoderLayer               ×16
  LlamaAttention                  ×16
  LlamaMLP                        ×16
  SiLUActivation                  ×16
  LlamaForCausalLM                ×1
  LlamaModel                      ×1


### Save and reload

The quantized model is still a standard HuggingFace `LlamaForCausalLM` —
every linear is a `LinearMXInt`, but the surrounding wiring is unchanged.
`save_pretrained` round-trips: quantized weights live in the state dict.

The reload-time catch: you need to **re-instantiate the `LinearMXInt`
modules first** so the state-dict shapes match. Run the same quantize
pass on a fresh fp16 baseline before `load_state_dict`.


In [ ]:
# Save
ckpt_dir = "output/llama32_mxint4_rtn"
model.save_pretrained(ckpt_dir)
tokenizer.save_pretrained(ckpt_dir)
print(f"saved to {ckpt_dir}")

# Reload 
#
#   _, fresh = setup_model("unsloth/Llama-3.2-1B", model_parallel=False,
#                          dtype=torch.float16, device=None,
#                          attn_implementation="eager")
#   pass_args = load_quant_config("output/recipe_a_rtn.toml")
#   fresh, _ = quantize_module_transform_pass(fresh, pass_args)
#   fresh.load_state_dict(torch.load(ckpt_dir + "/pytorch_model.bin"))
#

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.40s/it]

saved to output/llama32_mxint4_rtn


## Measure the accuracy — perplexity

WikiText perplexity is the cheapest sanity check: fast and high-signal.
We use the same `eval_ppl` CLI throughout this notebook — once for the
fp16 reference, then once per recipe.

### fp16 baseline


In [16]:
!python -m quant_eval.cli.eval_ppl \
    --model_name unsloth/Llama-3.2-1B \
    --dtype float16 \
    --seqlen 2048

/home/jn1020/projects/PLENA_Software/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
INFO     Set logging level to debug
Perplexity Evaluation
Model: unsloth/Llama-3.2-1B
Dataset: wikitext
Quantization: None (baseline)
INFO     Setting up model unsloth/Llama-3.2-1B with dtype torch.float16, device cuda:0, attn_implementation=sdpa
INFO     Tokenizer setup complete
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 146/146 [00:00<00:00, 338.83it/s]
INFO     Model setup complete
Token indices sequence length is longer than the specified maximum sequence length for this model (289077 > 131072). Running this sequence through the model will result in indexing errors
Evaluating PPL: 100%|

!!! note "Expected"
    `ppl ≈ 9.7` — the fp16 reference.

### Recipe A — MXInt4 RTN


In [17]:
!python -m quant_eval.cli.eval_ppl \
    --model_name unsloth/Llama-3.2-1B \
    --quant_config output/recipe_a_rtn.toml \
    --dtype float16 \
    --seqlen 2048

/home/jn1020/projects/PLENA_Software/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
INFO     Set logging level to debug
Perplexity Evaluation
Model: unsloth/Llama-3.2-1B
Dataset: wikitext
Quantization config: output/recipe_a_rtn.toml
INFO     Setting up model unsloth/Llama-3.2-1B with dtype torch.float16, device cuda:0, attn_implementation=eager
INFO     Tokenizer setup complete
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 146/146 [00:00<00:00, 388.14it/s]
INFO     Model setup complete
INFO     Quantizing 113 linear layers...
INFO     Quantization complete in 13.0s
=== Model Layers and Devices ===
: LlamaForCausalLM | device: cuda:0
model: LlamaModel | device: cuda:0
model.e

!!! note "Expected"
    `ppl ≈ 18.8` — round-to-nearest leaves a visible gap from fp16. The
    next two recipes close most of it.


## Recipe B — add `[gptq]`

GPTQ ([Frantar et al., 2022](https://arxiv.org/abs/2210.17323)) is a
one-time pre-pass: it walks the decoder layer by layer and rewrites
each linear's weight in place using Hessian-based error compensation
against a calibration set (we use wikitext2 here). Our `[gptq]` pass
extends the original integer-only formulation to MX data formats with
block-wise clipping search, as described in the PLENA paper.

The TOML adds a top-level `[gptq]` block and a `gptq = true` flag on
each selector so the linear classes consume the calibrated weights
instead of re-quantizing.

The `checkpoint_dir` enables **auto-resume**: re-running this cell only
recomputes layers that aren't already on disk. First run takes a few
minutes for Llama-3.2-1B; subsequent runs finish in seconds.


In [18]:
%%writefile output/recipe_b_gptq.toml
by = "regex_name"

[gptq]
model_name       = "unsloth/Llama-3.2-1B"
format           = "mxint"
dataset          = "wikitext2"
nsamples         = 32
seqlen           = 512
cali_batch_size  = 8
quantile_search  = true
clip_search_y    = true
checkpoint_dir   = "output/checkpoints/recipe_b_gptq"

    [gptq.weight_config]
    weight_block_size = 32
    weight_width      = 4

["model\\.layers\\.\\d+\\.self_attn\\.(q|k|v|o)_proj"]
name = "mxint"
weight_block_size = 32
weight_width = 4
data_in_block_size = 32
data_in_width = 4
gptq = true                # consume the calibrated weights

["model\\.layers\\.\\d+\\.mlp\\.(gate|up|down)_proj"]
name = "mxint"
weight_block_size = 32
weight_width = 4
data_in_block_size = 32
data_in_width = 4
gptq = true

Writing output/recipe_b_gptq.toml


In [19]:
!python -m quant_eval.cli.eval_ppl \
    --model_name unsloth/Llama-3.2-1B \
    --quant_config output/recipe_b_gptq.toml \
    --dtype float16 \
    --seqlen 2048

/home/jn1020/projects/PLENA_Software/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
INFO     Set logging level to debug
Perplexity Evaluation
Model: unsloth/Llama-3.2-1B
Dataset: wikitext
Quantization config: output/recipe_b_gptq.toml
INFO     Setting up model unsloth/Llama-3.2-1B with dtype torch.float16, device cuda:0, attn_implementation=eager
INFO     Tokenizer setup complete
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 146/146 [00:01<00:00, 136.87it/s]
INFO     Model setup complete
INFO     Quantizing 113 linear layers...
Token indices sequence length is longer than the specified maximum sequence length for this model (2436214 > 131072). Running this sequence through th

!!! note "Expected"
    `ppl ≈ 14.2` — GPTQ closes the RTN gap. We will target **activation quantization error** next.


## Recipe C — add `[rotation_search]`

Rotation search adds online Hadamard rotation to a subset of the matmuls,
following the QuaRot recipe ([Ashkboos et al., 2024](https://arxiv.org/abs/2404.00456))
for outlier-free low-bit inference. The greedy forward search tries
enabling rotation on each matmul type in turn, commits the one with the
largest ppl drop, and repeats. Winners are written to `cache_path` so
subsequent runs skip the search entirely. PLENA applies rotation
*selectively* — naive whole-model rotation degrades MX-quantized weights,
so the search isolates the matmuls where rotation actually helps.

The recipe **reuses Recipe B's GPTQ checkpoint** — `checkpoint_dir` points
at the same `output/checkpoints/recipe_b_gptq` directory, so we don't
redo GPTQ. Only the rotation search runs.

First execution takes a few minutes (one ppl forward per candidate matmul
type per round). Cached re-runs finish in seconds.


In [20]:
%%writefile output/recipe_c_rotsearch.toml
by = "regex_name"

[gptq]
model_name       = "unsloth/Llama-3.2-1B"
format           = "mxint"
dataset          = "wikitext2"
nsamples         = 32
seqlen           = 512
cali_batch_size  = 8
quantile_search  = true
clip_search_y    = true
checkpoint_dir   = "output/checkpoints/recipe_b_gptq"  

    [gptq.weight_config]
    weight_block_size = 32
    weight_width      = 4

[rotation_search]
calib_nsamples  = 32
calib_seqlen    = 512
improvement_eps = 0.0
cache_path      = "output/checkpoints/recipe_c_rotsearch/rotation_decisions.json"

["model\\.layers\\.\\d+\\.self_attn\\.(q|k|v|o)_proj"]
name = "mxint"
weight_block_size = 32
weight_width = 4
data_in_block_size = 32
data_in_width = 4
gptq = true

["model\\.layers\\.\\d+\\.mlp\\.(gate|up|down)_proj"]
name = "mxint"
weight_block_size = 32
weight_width = 4
data_in_block_size = 32
data_in_width = 4
gptq = true

Writing output/recipe_c_rotsearch.toml


In [21]:
!python -m quant_eval.cli.eval_ppl \
    --model_name unsloth/Llama-3.2-1B \
    --quant_config output/recipe_c_rotsearch.toml \
    --dtype float16 \
    --seqlen 2048

/home/jn1020/projects/PLENA_Software/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
INFO     Set logging level to debug
Perplexity Evaluation
Model: unsloth/Llama-3.2-1B
Dataset: wikitext
Quantization config: output/recipe_c_rotsearch.toml
INFO     Setting up model unsloth/Llama-3.2-1B with dtype torch.float16, device cuda:0, attn_implementation=eager
INFO     Tokenizer setup complete
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 146/146 [00:00<00:00, 435.18it/s]
INFO     Model setup complete
INFO     Quantizing 113 linear layers...
Token indices sequence length is longer than the specified maximum sequence length for this model (2436214 > 131072). Running this sequence throu

!!! note "Expected"
    `ppl ≈ 12.6` — rotation search closes most of the remaining gap.


## Recap — the progressive cost

| Recipe | What it adds              | WikiText ppl (Llama-3.2-1B) |
|--------|---------------------------|------------------------------|
| fp16   | (reference)               | ~9.7                         |
| A — RTN| MXInt4 round-to-nearest   | ~18.8                        |
| B — +GPTQ | Hessian-aware weights  | ~14.2                        |
| C — +Rotation | Per-matmul Hadamard | ~10.0–10.5                  |

Same regex selectors, same target precision. Three TOML files; the
recipe is the only thing that changes.


## Beyond perplexity

Perplexity is the fast iteration metric. Once you have a recipe you trust,
broader benchmarks decide whether it's actually good enough. PLENA's other
eval CLIs all consume the same `--quant_config`.

### Zero-shot tasks — lm-eval-harness

We run our best recipe (C) on three standard zero-shot tasks. `--limit 100`
caps each task at 100 samples for a quick read; drop it for a full run.


In [22]:
!python -m quant_eval.cli.eval_lm \
    --model_name unsloth/Llama-3.2-1B \
    --quant_config output/recipe_c_rotsearch.toml \
    --tasks arc_easy,hellaswag,winogrande \
    --batch_size 16 \
    --limit 100

/home/jn1020/projects/PLENA_Software/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
INFO     Set logging level to debug
lm-eval — fixed activation precision (no phase switch)
  Model  : unsloth/Llama-3.2-1B
  Tasks  : arc_easy,hellaswag,winogrande
  Weights: output/recipe_c_rotsearch.toml
  Seqlen : 2048
INFO     Setting up model unsloth/Llama-3.2-1B with dtype torch.bfloat16, device cuda:0, attn_implementation=eager
^C


### Code generation — HumanEval+

Requires the `evalplus` extra:

```bash
uv sync --extra evalplus
```


In [ ]:
!python -m quant_eval.cli.eval_evalplus \
    --model_name unsloth/Llama-3.2-1B \
    --quant_config output/recipe_c_rotsearch.toml \
    --dataset humaneval \
    --greedy --n_samples 1

## Going further

PLENA ships more eval surfaces than this tutorial covers:

- **`eval_phase_lm`** — different activation precision for prefill vs decode
  (and attention vs FFN). Useful for studying disaggregated-inference
  trade-offs.
- **`eval_phase_bfcl`** — function-calling (BFCL) under the same
  phase-dependent precision.
- **`eval_dllm` / `eval_llada`** — diffusion language models with
  block-diffusion sampling.
- **`eval_osworld`** — agentic desktop tasks via OSWorld.

See
[Evaluation commands](https://aicrosssim.github.io/PLENA_Software/reference/cli/) for the full reference.

For paper-table reproductions (Llama-2/3 across model sizes and bit
configs), see `plena_experiments/` in the repo root.


## Wrap-up

You just:

1. Wrote three quantization recipes — RTN, +GPTQ, +rotation_search.
2. Applied them to Llama-3.2-1B and watched perplexity improve at each
   step.
3. Used the same recipe to drive three independent benchmarks (ppl,
   lm-eval, HumanEval+).
4. Saved the quantized model as a standard HF checkpoint.


